<img src=https://www.factset.com/hubfs/Assets/images/factset-logo.svg width="300" align="left">


# FactSet ESG - Getting Started
This notebook demonstrates basic features of the FactSet ESG API by walking through the following steps:

1. Import Python packages and set up credentials
2. Define helper functions for API calls
3. For each ESG API endpoint, fetch data and display in a Pandas DataFrame

Additional Materials can be found on the [FactSet Developer Portal](https://developer.factset.com/api-catalog/factset-esg-api).

## 1. Import the required packages

In [ ]:
import requests
import json
import time
import pandas as pd
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
from pandas import json_normalize

import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
USERNAME = os.getenv("USERNAME")
APIKEY = os.getenv("APIKEY")

ESG_SCORES_URL = 'https://api.factset.com/content/factset-esg/v3/truvalue/scores'
ESG_SPOTLIGHTS_URL = 'https://api.factset.com/content/factset-esg/v3/truvalue/spotlights'
ESG_ARTICLES_URL = 'https://api.factset.com/content/factset-esg/v3/truvalue/articles'

## 2. Create a connection object

Enter your credentials for 'Username' and 'API Key' variables below.

To generate an API key, visit  **[Manage API Keys](https://developer.factset.com/factset/api-key-listing)**. Click [here](https://developer.factset.com/authentication) for more details on Authentication.

Place your Username and API key in a `.env` file:

In [ ]:
authorization = (USERNAME, APIKEY)
headers = {'Accept': 'application/json', 'Content-Type': 'application/json'}

## Helper functions

In [ ]:
def reorder_columns(df, leading_cols=("date", "requestId")):
    """Move leading_cols to the front of the DataFrame, keeping the rest in original order."""
    front = [c for c in leading_cols if c in df.columns]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]


def fetch_esg(url, request_body, label, date_col="date"):
    """Generic fetch for any FactSet ESG endpoint. Returns a cleaned DataFrame."""
    response = requests.post(
        url=url,
        data=json.dumps(request_body),
        auth=authorization,
        headers=headers,
        verify=False
    )
    print(f"{label} - HTTP Status: {response.status_code}")

    if response.status_code != 200:
        print(f"Error: {response.text[:300]}")
        return pd.DataFrame()

    df = json_normalize(response.json()['data'])
    df = reorder_columns(df, leading_cols=(date_col, "requestId"))
    print(f"Records: {len(df)}, Columns: {len(df.columns)}")
    time.sleep(0.15)
    return df

In [ ]:
# Quick test: PULSE scores for Amazon (TOPLEVEL only, single date)
pulse_toplevel_df = fetch_esg(
    url=ESG_SCORES_URL,
    request_body={
        "data": {
            "ids": ["AMZN-US"],
            "scoreType": "PULSE",
            "fields": ["TOPLEVEL"],
            "startDate": "2023-12-31",
            "endDate": "2023-12-31",
            "frequency": "M",
            "calendar": "FIVEDAY"
        }
    },
    label="PULSE (TOPLEVEL)",
)
display(pulse_toplevel_df)

# 3. FactSet ESG API (v3)

The notebook creates a requests object and displays a dataframe for each of the following ESG API endpoints:

1. **Truvalue Scores (PULSE)** - [/factset-esg/v3/truvalue/scores](#tvl_scores_pulse)
2. **Truvalue Scores (RANKS)** - [/factset-esg/v3/truvalue/scores](#tvl_scores_ranks)
3. **Truvalue Scores (All Fields)** - [/factset-esg/v3/truvalue/scores](#tvl_scores_all)
4. **Truvalue Spotlights** - [/factset-esg/v3/truvalue/spotlights](#tvl_spotlights)
5. **Truvalue Articles** - [/factset-esg/v3/truvalue/articles](#tvl_articles)

For additional details regarding each endpoint's request parameters or response models, visit the [FactSet ESG](https://developer.factset.com/api-catalog/factset-esg-api) specification page.

<a id='tvl_scores_pulse'></a>
## 3.1 Truvalue Scores (PULSE, INSIGHT, MOMENTUM)

FactSet Truvalue Labs Scores provides short-term, long-term, and momentum scores that are generated for 26 SASB categories defined by the Sustainability Accounting Standards Board. Use the `scoreType` parameter to select PULSE, INSIGHT, MOMENTUM, and other score types. Use `fields` to control the level of detail: TOPLEVEL, PILLARS, DIMENSIONS, or SASBCATEGORIES.

In [ ]:
# PULSE scores with all SASB categories for Amazon (monthly, full year)
pulse_sasb_df = fetch_esg(
    url=ESG_SCORES_URL,
    request_body={
        "data": {
            "ids": ["AMZN-US"],
            "scoreType": "PULSE",
            "fields": ["SASBCATEGORIES"],
            "startDate": "2023-01-01",
            "endDate": "2023-12-31",
            "frequency": "M",
            "calendar": "FIVEDAY"
        }
    },
    label="PULSE (SASBCATEGORIES)",
)
display(pulse_sasb_df)

<a id='tvl_scores_ranks'></a>
## 3.2 Truvalue Scores (RANKS)

In v3, Ranks are retrieved via the same `/truvalue/scores` endpoint using `scoreType: "RANKS"`. Ranks indicate if a company is a Leader, Above Average, Average, Below Average, or a Laggard, directly mapping from Industry Percentiles.

In [ ]:
# RANKS for Apple (monthly, full year)
ranks_df = fetch_esg(
    url=ESG_SCORES_URL,
    request_body={
        "data": {
            "ids": ["AAPL-USA"],
            "scoreType": "RANKS",
            "fields": ["TOPLEVEL"],
            "startDate": "2023-01-01",
            "endDate": "2023-12-31",
            "frequency": "M",
            "calendar": "FIVEDAY"
        }
    },
    label="RANKS (TOPLEVEL)",
)
display(ranks_df)

<a id='tvl_scores_all'></a>
## 3.3 Truvalue Scores (All Fields)

Retrieves scores with all field levels (TOPLEVEL, PILLARS, DIMENSIONS, SASBCATEGORIES) for the requested scoreType and ids. This provides a comprehensive view across all 26 SASB categories.

In [ ]:
# PULSE with all field levels for Apple (single date)
all_fields_df = fetch_esg(
    url=ESG_SCORES_URL,
    request_body={
        "data": {
            "ids": ["AAPL-USA"],
            "scoreType": "PULSE",
            "fields": ["TOPLEVEL", "PILLARS", "DIMENSIONS", "SASBCATEGORIES"],
            "startDate": "2023-12-31",
            "endDate": "2023-12-31",
            "frequency": "D",
            "calendar": "FIVEDAY"
        }
    },
    label="PULSE (All Fields)",
)
display(all_fields_df)

<a id='tvl_spotlights'></a>
## 3.4 Truvalue Spotlights

FactSet Truvalue Labs Spotlight Data solutions are a daily collection of the most important positive and negative ESG events detected by the algorithms, with quantitative metadata to enable timely and systematic trading strategies and portfolio management. Qualitative informational data points such as the headline and key bullet points for articles is also included.

In [ ]:
# Spotlights for Microsoft (Human Rights category)
spotlights_df = fetch_esg(
    url=ESG_SPOTLIGHTS_URL,
    request_body={
        "data": {
            "ids": ["MSFT-US"],
            "startDate": "2022-01-01",
            "endDate": "2023-10-30",
            "categories": ["HumanRightsAndCommunityRelations"],
            "fields": ["spotlightPillar", "tvGroupId"],
            "primaryOnly": True,
            "isRemoved": False
        }
    },
    label="Spotlights",
    date_col="liveDate",
)
display(spotlights_df)

<a id='tvl_articles'></a>
## 3.5 Truvalue Articles

Articles endpoint allows to retrieve underlying news articles used by the AI engine to calculate the ESG Scores of companies and therefore provides ESG relevant news and also transparency into the ESG Scores.

In [ ]:
# Articles for Amazon (Human Rights category)
articles_df = fetch_esg(
    url=ESG_ARTICLES_URL,
    request_body={
        "data": {
            "ids": ["AMZN-US"],
            "categories": ["HumanRightsAndCommunityRelations"],
            "fields": ["datePublication"],
            "startDate": "2023-01-01",
            "endDate": "2023-10-30",
            "dateOf": "PUBLICATION"
        }
    },
    label="Articles",
    date_col="datePublication",
)
display(articles_df)